# 15 · Workflows – agents that follow a plan

One agent with many tools gets confused. A **workflow** splits the job into steps, each with a focused agent, and defines **who goes next**. This chapter builds three patterns with **Microsoft Agent Framework**, which Foundry recommends for workflows (the visual workflow designer in the portal is being retired after 2026-12-01; YAML and code workflows keep running, e.g. as hosted agents).

| 🔀 Pattern | 🧩 Shape | 🎯 Use for | 🏃 Here |
|---|---|---|---|
| **Sequential** | A → B → C | pipelines, document processing | ✅ invoice pipeline |
| **Group chat** | star: an orchestrator picks the next speaker | review boards, debate | ✅ review board |
| **Handoff** | agents pass control by routing rules, interactive | triage, specialists | ✅ run with scripted user turns |
| Concurrent / Magentic | parallel / manager-led | fan-out, open-ended tasks | mentioned |
| **Declarative YAML** | the same flow in a file | reviewable, versioned flows | ✅ [workflows/invoice-pipeline.yaml](../workflows/invoice-pipeline.yaml) |

These run **in your Python process** (local, client side), calling Foundry models. To run a workflow *in Foundry*, wrap it as an agent and host it like chapter 09. Deep dive: [docs/workflows.md](../docs/workflows.md).

In [ ]:
import sys; sys.path.insert(0, '.')
import importlib, stark_helpers; importlib.reload(stark_helpers)   # pick up helper changes without a kernel restart
from stark_helpers import *
from stark_docs import *

fix_dotnet_for_powerfx()   # only changes anything for an x64 Python on Windows-on-ARM (needed in section 4)

import re
from typing import Annotated
from pydantic import Field
from agent_framework import Agent, AgentResponseUpdate, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework.orchestrations import SequentialBuilder, GroupChatBuilder, GroupChatState, HandoffBuilder

client = FoundryChatClient(project_endpoint=cfg("STARK_PROJECT_ENDPOINT"), model=model("chat"), credential=credential())

## Tools for the pipeline
Plain Python functions. `approval_mode="never_require"` lets the agent call them without a human (fine for read-only tools).

In [ ]:
@tool(approval_mode="never_require")
def extract_invoice_fields(path: Annotated[str, Field(description="Invoice file, e.g. data/docs/invoice-pym-4471.pdf")]) -> str:
    """Read an invoice with Document Intelligence and return fields, line items and confidences as JSON."""
    import json
    return json.dumps(extract_invoice(path), default=str)

LIMITS = {"default": 5000, "pepper potts": 250000, "tony stark": 1000000}

@tool(approval_mode="never_require")
def check_approval_limit(amount: Annotated[float, Field(description="Invoice total in USD")],
                         approver: Annotated[str, Field(description="e.g. 'pepper potts'")]) -> str:
    """Check whether an amount is within the approver's limit."""
    limit = LIMITS.get(approver.strip().lower(), LIMITS["default"])
    return f"limit={limit} USD, within_limit={amount <= limit}"

skill_body = re.sub(r"^---.*?---\n", "", (ROOT / "skills" / "invoice-review" / "SKILL.md").read_text(encoding="utf-8"), flags=re.S)

## 1 · Sequential: the invoice pipeline
Three specialists. The reviewer gets the **invoice-review skill** (chapter 07) injected as instructions. `intermediate_output_from` makes the first two steps visible, not only the last.

In [ ]:
extractor = client.as_agent(name="extractor", tools=[extract_invoice_fields],
    instructions="Extract the invoice. Call extract_invoice_fields with the path from the user, then list vendor, number, date, total and any low-confidence (<0.8) field.")
reviewer = client.as_agent(name="reviewer", tools=[check_approval_limit],
    instructions="You review the extracted invoice. Follow this checklist exactly:\n" + skill_body +
                 "\nUse check_approval_limit with approver 'pepper potts'.")
reporter = client.as_agent(name="reporter",
    instructions="Write a three-bullet message for Pepper Potts: what was reviewed, the decision, what she must do next.")

def build_pipeline():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    return SequentialBuilder(participants=[extractor, reviewer, reporter], intermediate_output_from=[extractor, reviewer]).build()

async def show(workflow, task):
    last = None
    async for event in workflow.run(task, stream=True):
        if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
            u = event.data
            if u.author_name != last:
                print(f"\n\n[{'FINAL' if event.type == 'output' else 'step'}] {u.author_name}: ", end="")
                last = u.author_name
            print(u.text, end="", flush=True)
    print()

await show(build_pipeline(), "Process the invoice at data/docs/invoice-pym-4471.pdf")

Run it again with `data/docs/invoice-oscorp-9930.pdf` (about 99,792 USD): the same flow now ends in a different decision, because the checklist and the limit tool decide, not luck.

> ⚠️ Build a **new workflow for every run** (`build_pipeline()`). A workflow object keeps its conversation, and running the same object a second time fails with `No tool call found for function call output`. The agents can be reused.

In [ ]:
await show(build_pipeline(), "Process the invoice at data/docs/invoice-oscorp-9930.pdf")

## 2 · Group chat: the review board
Agents share one conversation; a **selector** picks the next speaker (here round-robin; use `orchestrator_agent=` for an LLM that decides). A termination condition ends the chat.

In [ ]:
engineer = client.as_agent(name="engineer", instructions="You are a suit engineer. Judge technical need in two sentences.")
accountant = client.as_agent(name="accountant", instructions="You are the accountant. Judge cost and budget in two sentences.")
security = client.as_agent(name="security", instructions="You are the security officer. Judge supply-chain and safety risk in two sentences. If all three of you agree, end with 'DECISION: <approve|reject>'.")

def round_robin(state: GroupChatState) -> str:
    names = list(state.participants.keys())
    return names[state.current_round % len(names)]

board = GroupChatBuilder(
    participants=[engineer, accountant, security],
    termination_condition=lambda conversation: sum(1 for m in conversation if m.role == "assistant") >= 6,
    intermediate_output_from=[engineer, accountant, security],
    selection_func=round_robin,
).build()

await show(board, "Should we buy 4 vials of Pym particles for 3,332 USD for the shrink-ray prototype?")

## 3 · Handoff: triage and specialists
In a **handoff**, agents decide to pass control based on the conversation; the user stays in the loop. `triage` routes each message to a specialist, the specialist answers, and control can go back to `triage`.

> ⚠️ Every participant must be created with `require_per_service_call_history_persistence=True`. Without it `build()` raises `Handoff workflows require all participant agents to have ...`. The flag keeps each agent's local history consistent with the service when control is handed over.

`enable_return_to_previous(False)` makes `triage` look at **every** user message. Without it, the next message goes straight to whoever spoke last.

In [ ]:
# handoff workflows need every participant to keep its history per service call
triage = client.as_agent(name="triage", require_per_service_call_history_persistence=True, instructions="Route the user: suits and hardware -> suit_engineer; access, badges, threats -> security_officer. Otherwise answer yourself.")
suit_engineer = client.as_agent(name="suit_engineer", require_per_service_call_history_persistence=True, instructions="You fix suits. Be practical. Hand back to triage when done.")
security_officer = client.as_agent(name="security_officer", require_per_service_call_history_persistence=True, instructions="You handle security incidents. Be strict. Hand back to triage when done.")

def build_handoff():
    # a workflow keeps its conversation between runs, so build a fresh one for every run
    return (
        HandoffBuilder(name="workshop_support", participants=[triage, suit_engineer, security_officer])
        .with_start_agent(triage)
        .enable_return_to_previous(False)
        .add_handoff(triage, [suit_engineer, security_officer])
        .add_handoff(suit_engineer, [triage])
        .add_handoff(security_officer, [triage])
        .build()
    )

print("handoff workflow built:", build_handoff().name)

### Run it with scripted user turns
A handoff workflow **pauses and asks the user** after each specialist answers: you get a `request_info` event with a `HandoffAgentUserRequest`. You answer it with `HandoffAgentUserRequest.create_response(text)`, or end the chat with `HandoffAgentUserRequest.terminate()`. The helper below plays a list of user messages instead of calling `input()`, and prints who speaks, so you can see the handoffs.

In [ ]:
from agent_framework.orchestrations import HandoffAgentUserRequest

async def chat(workflow, user_turns):
    """Run a handoff workflow with scripted user turns; print who speaks and where control goes."""
    turns = list(user_turns)
    print("USER:", turns[0])
    responses = None
    first = turns.pop(0)
    while True:
        pending, last = [], None
        stream = workflow.run(first, stream=True) if responses is None else workflow.run(responses=responses, stream=True)
        async for event in stream:
            if event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
                pending.append(event)
            elif event.type == "output" and isinstance(event.data, AgentResponseUpdate) and event.data.text:
                if event.data.author_name != last:
                    print(f"\n[{event.data.author_name}] ", end="")
                    last = event.data.author_name
                print(event.data.text, end="", flush=True)
        print()
        if not pending:
            break
        if turns:
            nxt = turns.pop(0)
            print("\nUSER:", nxt)
            responses = {r.request_id: HandoffAgentUserRequest.create_response(nxt) for r in pending}
        else:
            responses = {r.request_id: HandoffAgentUserRequest.terminate() for r in pending}
        first = None

In [ ]:
await chat(build_handoff(), [
    "My Mark 42 repulsor is jammed and will not fire.",
    "Thanks. Now someone used a stolen badge to enter the vault yesterday.",
    "Thanks, that is all.",
])

Read the speaker labels: a hardware question goes to `suit_engineer`, a badge question to `security_officer`, and the closing thanks is answered by `triage`. An agent may sometimes answer a question itself, so the exact route can vary between runs. For a real chat, replace the list with `input(...)`.

## 4 · The same pipeline as YAML
Declarative workflows keep the *flow* in a reviewable file; agents are registered by name. (Needs `agent-framework-declarative`, Python 3.10–3.13 and a **.NET runtime**: expressions like `=System.ConversationId` are evaluated by PowerFx. If you see `PowerFx is not available (dotnet runtime not installed)` although `dotnet` is installed, your Python and your .NET have different CPU architectures; `fix_dotnet_for_powerfx()` in the first cell handles the common x64-Python-on-ARM case. Restart the kernel after changing it.)

In [ ]:
print((ROOT / "workflows" / "invoice-pipeline.yaml").read_text())

In [ ]:
from agent_framework.declarative import WorkflowFactory
factory = WorkflowFactory()
factory.register_agent("ExtractorAgent", extractor)
factory.register_agent("ReviewerAgent", reviewer)
factory.register_agent("ReporterAgent", reporter)
declared = factory.create_workflow_from_yaml_path(ROOT / "workflows" / "invoice-pipeline.yaml")
result = await declared.run("Process the invoice at data/docs/invoice-pym-4471.pdf")
for out in result.get_outputs():
    print(out)

## 5 · From notebook to Foundry
A workflow can look like an agent: `workflow.as_agent(name="Invoice Pipeline Agent")`. Hand that to `ResponsesHostServer` (see [src/jarvis-hosted/main.py](../src/jarvis-hosted/main.py)), deploy with `azd deploy`, and the pipeline gets an endpoint, identity, tracing – and can be **evaluated** (chapter 10) like any agent.

## Recap
| Need | Pattern |
|---|---|
| Fixed steps | sequential |
| Debate / multiple viewpoints | group chat |
| Route to the right specialist, user in the loop | handoff |
| Reviewable flow in git | declarative YAML |
| Run in Foundry | `as_agent` + hosted agent |

> 💡 Next: [16 · Vision – agents calling agents](16_vision_a2a_capstone.ipynb) 🧪.